# 03 - Integration Analysis

Notebook de apoyo para revisar datasets Silver integrados. No transforma datos, no exporta archivos y no construye Gold.

Las celdas leen salidas locales si existen bajo `data/silver/integrated`. Si todavía no se generaron, muestran mensajes controlados.

## Preparación del entorno

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INTEGRATED_DIR = PROJECT_ROOT / 'data' / 'silver' / 'integrated'
INTEGRATED_DIR

## Carga controlada de datasets integrados

In [ ]:
def read_integrated_dataset(dataset_name: str):
    """Lee un dataset integrado si existe."""

    path = INTEGRATED_DIR / dataset_name
    if not path.exists():
        print(f'No existe {path}. Ejecuta primero la integración Silver.')
        return None

    return pd.read_parquet(path)


bridge_df = read_integrated_dataset('municipal_entity_bridge')
mef_df = read_integrated_dataset('mef_municipal_amounts')
predial_df = read_integrated_dataset('predial_entity_period')
renamu_df = read_integrated_dataset('renamu_municipal_context')
coverage_df = read_integrated_dataset('integration_coverage')

## Cobertura general

In [ ]:
coverage_df if coverage_df is not None else pd.DataFrame()

## Puente sec_ejec -> ubigeo

In [ ]:
if bridge_df is None:
    bridge_summary_df = pd.DataFrame()
else:
    bridge_summary_df = pd.DataFrame([
        {
            'bridge_rows': len(bridge_df),
            'sec_ejec_distinct': bridge_df['sec_ejec'].dropna().nunique() if 'sec_ejec' in bridge_df else 0,
            'ubigeo_distinct': bridge_df['ubigeo'].dropna().nunique() if 'ubigeo' in bridge_df else 0,
            'valid_ubigeo_rows': int(bridge_df.get('is_valid_ubigeo', pd.Series(dtype=bool)).fillna(False).sum()),
            'renamu_match_rows': int(bridge_df.get('has_renamu_match', pd.Series(dtype=bool)).fillna(False).sum()),
        }
    ])

bridge_summary_df

## MEF con y sin puente

In [ ]:
if mef_df is None or bridge_df is None or 'sec_ejec' not in mef_df or 'sec_ejec' not in bridge_df:
    mef_match_df = pd.DataFrame()
else:
    bridge_sec_ejec = set(bridge_df['sec_ejec'].dropna().astype(str))
    mef_sec_ejec = mef_df['sec_ejec'].dropna().astype(str)
    mef_match_df = pd.DataFrame([
        {
            'mef_sec_ejec_distinct': mef_sec_ejec.nunique(),
            'with_bridge': mef_sec_ejec[mef_sec_ejec.isin(bridge_sec_ejec)].nunique(),
            'without_bridge': mef_sec_ejec[~mef_sec_ejec.isin(bridge_sec_ejec)].nunique(),
        }
    ])

mef_match_df

## Predial y RENAMU con y sin match

In [ ]:
if bridge_df is None or renamu_df is None:
    territory_match_df = pd.DataFrame()
else:
    predial_ubigeos = set(bridge_df.get('ubigeo', pd.Series(dtype=str)).dropna().astype(str))
    renamu_ubigeos = set(renamu_df.get('ubigeo', pd.Series(dtype=str)).dropna().astype(str))
    territory_match_df = pd.DataFrame([
        {
            'predial_ubigeos': len(predial_ubigeos),
            'renamu_ubigeos': len(renamu_ubigeos),
            'intersection': len(predial_ubigeos & renamu_ubigeos),
            'predial_without_renamu': len(predial_ubigeos - renamu_ubigeos),
            'renamu_without_predial': len(renamu_ubigeos - predial_ubigeos),
        }
    ])

territory_match_df

## Resumen de granularidades

In [ ]:
grain_rows = []

for name, df in {
    'municipal_entity_bridge': bridge_df,
    'mef_municipal_amounts': mef_df,
    'predial_entity_period': predial_df,
    'renamu_municipal_context': renamu_df,
}.items():
    if df is None:
        continue
    grain_rows.append(
        {
            'dataset': name,
            'rows': len(df),
            'columns': len(df.columns),
            'integration_grain_values': sorted(df['integration_grain'].dropna().unique().tolist()) if 'integration_grain' in df else [],
        }
    )

pd.DataFrame(grain_rows)

## Interpretación de riesgos

- MEF se mantiene agregado por recurso, año, mes, sec_ejec y clasificadores presupuestales para evitar joins crudos con fuentes de distinta granularidad.
- Predial conserva granularidad de formulario y tiempo estadístico en `predial_entity_period`.
- RENAMU funciona como contexto territorial por `ubigeo`.
- El puente `sec_ejec -> ubigeo` debe validarse antes de usarlo como dimensión definitiva.
- Las métricas de cobertura informan preparación para integración; no son KPIs Gold.